In [ ]:
# Run in Colab cell
!pip install -q pypdf sentence-transformers faiss-cpu pypdf2 llama-cpp-python gradio
# If you need OCR for scanned PDFs:
!apt-get install -qq tesseract-ocr libtesseract-dev
!pip install -q pytesseract pillow


In [ ]:
from pypdf import PdfReader
import os

def pdf_to_text(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)

# Convert all uploaded PDFs and write combined TXT file
all_text = ""
for fname in os.listdir("."):
    if fname.lower().endswith(".pdf"):
        print("Converting", fname)
        t = pdf_to_text(fname)
        all_text += f"\n\n--- FILE: {fname} ---\n\n" + t

with open("converted_notes.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print("Saved converted_notes.txt, length:", len(all_text))


In [ ]:
!apt-get install -qq poppler-utils


In [ ]:
!pip install -q pdf2image


In [ ]:
# Run only if pages return empty text (scanned images)
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

def pdf_to_text_ocr(pdf_path, dpi=200):
    pages = convert_from_path(pdf_path, dpi=dpi)
    text_pages = []
    for i, page in enumerate(pages):
        txt = pytesseract.image_to_string(page)
        text_pages.append(txt)
    return "\n".join(text_pages)

# Example usage:
# text = pdf_to_text_ocr("scanned_notes.pdf")


In [ ]:
from pdf2image import convert_from_path
print("✅ pdf2image is working!")


In [ ]:
from google.colab import files
files.upload()


In [ ]:
import os
os.listdir()


In [ ]:
from pypdf import PdfReader

pdf_path = "Curriculum of ai.pdf"   # ✅ exact file name from your folder
reader = PdfReader(pdf_path)

text = ""
for page in reader.pages:
    txt = page.extract_text()
    if txt:
        text += txt + "\n"

print("✅ Text length:", len(text))
print("✅ First 300 characters:\n", text[:300])


In [ ]:
def chunk_text_safe(text, min_words=100, max_words=250, overlap_words=40):
    words = text.split()
    n = len(words)

    print("✅ Total words in document:", n)

    if n < min_words:
        print("⚠️ Text too short — returning single chunk")
        return [" ".join(words)]

    chunks = []
    i = 0
    while i < n:
        end = min(i + max_words, n)
        chunk = " ".join(words[i:end])
        chunks.append(chunk)
        i = end - overlap_words
        if i < 0:
            i = 0

    return chunks

chunks = chunk_text_safe(text)

print("✅ Number of chunks:", len(chunks))
print("✅ Example chunk:\n", chunks[0][:400])


In [ ]:
with open("converted_notes.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ converted_notes.txt updated")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# ✅ Choose fast + accurate embedding model
embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

def embed_texts_safe(text_list, batch_size=64):
    if len(text_list) == 0:
        raise ValueError("❌ No chunks found! Cannot generate embeddings.")

    print("✅ Generating embeddings for", len(text_list), "chunks...")
    embs = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        e = embedder.encode(
            batch,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True   # ✅ important for cosine similarity
        )
        embs.append(e)

    embs = np.vstack(embs)
    print("✅ Embeddings generated successfully!")
    return embs

# ✅ Generate embeddings safely
embeddings = embed_texts_safe(chunks)

print("✅ Embeddings shape:", embeddings.shape)



In [ ]:
import faiss

d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)   # cosine similarity
index.add(embeddings)

print("✅ FAISS index created with", index.ntotal, "vectors")


In [ ]:
import faiss
import pickle
import numpy as np

# ✅ Safety check
if embeddings is None or len(embeddings) == 0:
    raise ValueError("❌ Embeddings are empty! Cannot build FAISS index.")

# ✅ Ensure float32 (FAISS requires this)
embeddings = embeddings.astype("float32")

# ✅ Get embedding dimension
d = embeddings.shape[1]

# ✅ Create FAISS index for COSINE similarity
index = faiss.IndexFlatIP(d)

# ✅ Normalize vectors ONCE for cosine similarity
faiss.normalize_L2(embeddings)

# ✅ Add vectors to FAISS
index.add(embeddings)

print("✅ FAISS index created")
print("✅ Total vectors stored:", index.ntotal)
print("✅ Vector dimension:", d)

# ✅ Save FAISS index to disk
faiss.write_index(index, "faiss_index.idx")

# ✅ Save text chunks (metadata)
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("✅ Files saved successfully:")
print("   - faiss_index.idx")
print("   - chunks.pkl")


In [ ]:
import faiss, pickle

index = faiss.read_index("faiss_index.idx")

with open("chunks.pkl", "rb") as f:
    loaded_chunks = pickle.load(f)

print("✅ Loaded FAISS vectors:", index.ntotal)
print("✅ Loaded chunks:", len(loaded_chunks))
print("✅ Sample chunk:\n", loaded_chunks[0][:300])


In [ ]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ✅ Load FAISS index
index = faiss.read_index("faiss_index.idx")

# ✅ Load chunks (metadata)
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

# ✅ Load embedding model again
embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

print("✅ FAISS vectors:", index.ntotal)
print("✅ Total chunks:", len(chunks))

# ✅ Retriever function
def retrieve(query, top_k=4):
    if not query.strip():
        return []

    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    D, I = index.search(q_emb, top_k)

    results = []
    for idx, score in zip(I[0], D[0]):
        results.append({
            "chunk": chunks[idx],
            "score": float(score)
        })

    return results


# ✅ Quick retrieval test
query_test = "What is artificial intelligence?"
results = retrieve(query_test)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (Score: {r['score']:.3f}) ---")
    print(r["chunk"][:300])


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!ollama pull phi3

In [ ]:
import requests

def ask_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    data = {"model": "phi3", "prompt": prompt, "stream": False}
    res = requests.post(url, json=data)
    return res.json()["response"]

print(ask_ollama("Explain Artificial Intelligence in simple words."))


In [ ]:
!nohup ollama serve &

In [ ]:
import faiss
import pickle
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

# ✅ Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ✅ Load FAISS index + chunks
index = faiss.read_index("faiss_index.idx")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

# ✅ Ask Ollama
def ask_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    data = {
        "model": "phi3",
        "prompt": prompt,
        "stream": False
    }
    res = requests.post(url, json=data)
    return res.json()["response"]

# ✅ Retrieve top K chunks
def retrieve_chunks(question, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, idx = index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]

# ✅ Full RAG pipeline
def ask_rag(question):
    context = "\n\n".join(retrieve_chunks(question))

    prompt = f"""
You are a helpful AI tutor.
Use only the context below to answer.

Context:
{context}

Question:
{question}

Answer:
"""
    return ask_ollama(prompt)

# ✅ TEST YOUR BOT
question = "Explain the syllabus of artificial intelligence"
answer = ask_rag(question)
print("\nAI Answer:\n", answer)


In [ ]:
import requests

def ask_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    data = {"model": "phi3", "prompt": prompt, "stream": False}
    res = requests.post(url, json=data)
    return res.json()["response"]

print(ask_ollama("Explain Artificial Intelligence in simple words."))


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Load FAISS index + chunks
index = faiss.read_index("faiss_index.idx")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

# Retrieve relevant chunks
def retrieve_chunks(question, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, idx = index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]

# Full RAG + LLM pipeline
def ask_rag(question):
    context = "\n\n".join(retrieve_chunks(question))

    prompt = f"""
You are a helpful AI tutor.
Use only the context below to answer.

Context:
{context}

Question:
{question}

Answer:
"""
    return ask_ollama(prompt)

# ✅ Test
question = "Explain the syllabus of artificial intelligence"
answer = ask_rag(question)
print("\nAI Answer:\n", answer)


In [ ]:
import requests

def ask_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    data = {"model": "phi3", "prompt": prompt, "stream": False}
    res = requests.post(url, json=data)
    return res.json()["response"]
